# autoprof

> Functionality for running autoprof to measure the ICL in images of galaxy clusters.

In [ ]:
# | default_exp autoprof

In [ ]:
# | exporti

import logging
import os
import re
from pathlib import Path
import subprocess
from textwrap import dedent

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import CCDData
from astropy.stats import sigma_clip
from astropy.visualization import simple_norm
from astropy.wcs import WCS
from matplotlib.patches import Circle, Ellipse
from photutils.aperture import CircularAnnulus, EllipticalAnnulus

from nicl.background import get_background

In [ ]:
# | export


def create_bkgsub_clean_images(
    image_filenames: str
    | Path
    | list[str]
    | list[Path],  # single image filename or list of image filenames
    output_dir: str
    | Path,  # output directory for background subtracted and/or cleaned images
    mask_filename: str | Path | None = None,  # and optional background mask filename
    output_background_dir: str
    | Path
    | None = None,  # optional output directory for background maps
    box_size=None,  #  box size for background estimation; if False or None, no background subtraction is performed
    filter_size=1,  # filter size for background estimation
    clean_nans=False,  # if True, NaNs are set to 99
):
    """Prepare background-subtracted and cleaned images for AutoProf."""
    if mask_filename is not None:
        mask = fits.getdata(mask_filename).astype(bool)
    else:
        mask = None

    if isinstance(image_filenames, str):
        image_filenames = [image_filenames]
    output_filenames = []

    for image_filename in image_filenames:
        image_filename = Path(image_filename)
        output_filename = image_filename.stem
        image = CCDData.read(image_filename, unit="adu")

        if box_size:
            bkg = get_background(
                image.data,
                mask=mask,
                box_size=box_size,
                filter_size=filter_size,
            )
            if output_background_dir is not None:
                output_background_dir = Path(output_background_dir)
                output_background_dir.mkdir(parents=True, exist_ok=True)
                bkg_filename = output_background_dir / f"{output_filename}_bkg.fits"
                fits.writeto(
                    bkg_filename,
                    bkg.background,
                    header=image.wcs.to_header(),
                    overwrite=True,
                )
            image.data -= bkg.background
            output_filename += "_bkgsub"

        if clean_nans:
            image.data[~np.isfinite(image.data)] = 99
            output_filename += "_cleaned"

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        output_filename = (output_dir / output_filename).with_suffix(".fits")
        image.write(output_filename, overwrite=True)
        output_filenames.append(output_filename)

    if len(output_filenames) == 1:
        return output_filenames[0]
    else:
        return output_filenames

In [ ]:
# | export


def run_autoprof(
    name,  # name of the object
    image_filename: Path | str,  # path to the image file
    mask_filename: Path | str | None = None,  # path to the mask file
    unit_type: str = "intensity",  # unit of the image
    gscale: float = 0.4,  # geometric scale
    pixelscale: float = 0.3,  # pixel scale
    zeropoint: float = 23.9,  # zeropoint
    background: float = 0,  # fixed background level; if None, the background level is estimated
    background_noise: float
    | None = None,  # fixed background noise level; if None, the background noise level is estimated
    out_dir: Path | str = "./",  # output directory
    fourier_orders: tuple[int, ...] | None = None,  # Fourier orders to measure
    fixed_centre: SkyCoord
    | bool
    | None = None,  # if True, fix to the image centre; if a SkyCoord, use the provided position
    forced_photometry: bool = False,  # if True, use the forced photometry pipeline
    forced_profile_filter: str | None = None,  # filter to use for the forced photometry
    forced_profile_filename: Path
    | str
    | None = None,  # path to the forced photometry profile file
):
    """Run AutoProf with optional forced photometry."""
    logger = logging.getLogger(__name__)
    os.makedirs(out_dir, exist_ok=True)
    config_filename = f"config-{name}.py"
    config_file = os.path.join(out_dir, config_filename)
    if forced_photometry:
        mode = "forced image"
        if not forced_profile_filename:
            raise ValueError(
                "forced_profile_filename must be provided when forced_photometry=True."
            )
        if not forced_profile_filter:
            raise ValueError(
                "forced_profile_filter must be provided when forced_photometry=True."
            )
        pipeline_steps = [
            "background",
            "psf",
            "center forced",
            "isophoteinit forced",
            "isophotefit forced",
            "isophoteextract forced",
            "writeprof",
        ]
    else:
        mode = "image"
        pipeline_steps = [
            "background",
            "psf",
            "center",
            "isophoteinit",
            "isophotefit",
            "isophoteextract",
            "writeprof",
        ]
    # Background subtraction is included, but the background level can be added back in later.

    if mask_filename is not None:
        pipeline_steps.insert(0, "mask segmentation map")

    # Write AutoProf config
    with open(config_file, "w") as f:
        f.write(
            dedent(f"""
                    ap_process_mode = '{mode}'
                    ap_name = '{name}'
                    ap_image_file = '{image_filename}'
                    ap_saveto = '{out_dir}'
                    ap_pixscale = {pixelscale}
                    ap_zeropoint = {zeropoint}
                    ap_samplegeometricscale = {gscale}
                    ap_doplot = True
                    ap_extractfull = True
                    ap_fluxunits = '{unit_type}'
                    ap_isoclip = True
                    ap_isoclip_nsigma = 5
                    ap_fit_limit = 1
                    ap_new_pipeline_steps = {pipeline_steps}
                    """)
        )
        if mask_filename is not None:
            f.write(f"ap_mask_file = '{mask_filename}'\n")
        if background is not None:
            f.write(f"ap_set_background = {background}\n")
        if background_noise is not None:
            f.write(f"ap_set_background_noise = {background_noise}\n")
        if fixed_centre:
            wcs = WCS(fits.getheader(image_filename))
            if isinstance(fixed_centre, SkyCoord):
                centre_pixels = wcs.world_to_pixel(fixed_centre)
            elif fixed_centre is True:
                centre_pixels = (np.array(wcs.pixel_shape) - 1) / 2
            centre_pixels = dict(zip(("x", "y"), centre_pixels))
            centre_pixels = {k: float(v) for k, v in centre_pixels.items()}
            f.write(f"ap_set_centre = {centre_pixels}\n")
        if forced_photometry:
            f.write(f"ap_forcing_profile = '{forced_profile_filename}'\n")
        if fourier_orders:
            f.write(f"ap_iso_measurecoefs = {fourier_orders}\n")

    # Run AutoProf
    log_filename = Path(out_dir) / f"autoprof-{name}.log"
    env = os.environ.copy()
    try:
        del env["MPLBACKEND"]
    except KeyError:
        pass
    process = subprocess.run(
        ["autoprof", config_filename, log_filename],
        cwd=out_dir,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

    if process.returncode == 0:
        logger.info(f"AutoProf completed successfully for {name}!")
    else:
        logger.error(f"AutoProf failed for {name}.")
        raise RuntimeError(f"AutoProf failed, see {log_filename} for details.")

In [ ]:
# | export


def get_autoprof_info(profile_filename):
    profile_filename = Path(profile_filename)
    with open(profile_filename.with_suffix(".aux")) as f:
        content = f.read()
    info = {}
    background_match = re.search(
        r"background:\s*([-\d.e+]+)\s*\+\-\s*([\d.e+-]+)", content
    )
    background_value = float(background_match.group(1))
    noise_match = re.search(r"noise:\s*([-\d.e+]+)", content)
    noise_value = float(noise_match.group(1))
    info["noise"] = noise_value
    pixscale_match = re.search(r"option ap_pixscale:\s*([\d.e+-]+)", content)
    pixscale = float(pixscale_match.group(1))
    info["pixscale"] = pixscale
    info["background_level"] = background_value / pixscale**2
    centre_match = re.search(r"center x:\s*([\d.]+) pix, y:\s*([\d.]+)", content)
    info["centre"] = [float(g) for g in centre_match.groups()]
    central_sb_match = re.search(r"central surface brightness:\s*([\d.]+)", content)
    info["central_sb"] = float(central_sb_match.group(1))
    zeropoint_match = re.search(r"ap_zeropoint:\s*([\d.]+)", content)
    info["zeropoint"] = float(zeropoint_match.group(1))
    fit_limit_match = re.search(r"fit limit semi-major axis:\s*([\d+]+)", content)
    info["fit_limit"] = float(fit_limit_match.group(1))
    return info

In [ ]:
# | export


def measure_surface_brightness_using_autoprof_isophotes(
    image_filename: Path | str,
    object_mask_filename: Path | str | None = None,
    profile_filename: Path | str | None = None,
    centre=None,  # centre of the profile; if None, the image centre is used
    annuli_shape="elliptical",
    pixelscale=0.3,
    rad_limit_annulus=None,
    plot_filename=None,
):
    """Extract surface-brightness profiles from an image using the output from AutoProf."""
    profile = pd.read_csv(profile_filename, skiprows=1)

    ccd = CCDData.read(image_filename, unit="adu")
    image = ccd.data
    wcs = ccd.wcs

    if centre is not None:
        x, y = wcs.world_to_pixel(centre)
    else:
        x, y = (np.array(wcs.pixel_shape) - 1) / 2

    mask = ~np.isfinite(image)
    if object_mask_filename is not None:
        object_mask = fits.getdata(object_mask_filename).astype(bool)
        mask |= object_mask
    masked_image = np.where(mask, np.nan, image)

    profile["R"] = profile["R"] / pixelscale  # Convert arcsec to pixels

    if rad_limit_annulus:
        profile = profile[profile["R"] < rad_limit_annulus]

    rad = profile["R"].values
    ellip = profile["ellip"].values
    pa = profile["pa"].values

    selected_annuli = sorted(set(rad))
    flux_stats = []
    problematic_annuli = []

    for i in range(len(selected_annuli) - 1):
        r_inner = selected_annuli[i]
        r_outer = selected_annuli[i + 1]
        e = ellip[i]
        # Convert from PA to photutils angle (the definition of theta is different in AutoProf and photutils/matplotlib)
        theta = np.deg2rad(pa[i]) - np.pi / 2

        if annuli_shape == "circular":
            annulus = CircularAnnulus((x, y), r_in=r_inner, r_out=r_outer)
        else:
            annulus = EllipticalAnnulus(
                (x, y),
                a_in=r_inner,
                a_out=r_outer,
                b_in=r_inner * (1 - e),
                b_out=r_outer * (1 - e),
                theta=theta,
            )
        annulus_mask = annulus.to_mask(method="center")
        mask_image = annulus_mask.to_image(masked_image.shape, dtype=bool)

        valid_flux_values = masked_image[np.isfinite(masked_image) & mask_image]

        total_pixels = np.sum(mask_image)
        total_valid = len(valid_flux_values)
        total_masked = total_pixels - total_valid

        if total_valid == 0:
            problematic_annuli.append(
                {"x": x, "y": y, "r_inner": r_inner, "r_outer": r_outer}
            )
            mean_flux = median_flux = std_flux = np.nan
            clipped_mean_flux = clipped_median_flux = clipped_std_flux = np.nan
        else:
            mean_flux = np.mean(valid_flux_values) / (pixelscale**2)
            median_flux = np.median(valid_flux_values) / (pixelscale**2)
            std_flux = (
                np.std(valid_flux_values) if len(valid_flux_values) > 1 else np.nan
            )

            clipped = sigma_clip(
                valid_flux_values, sigma=3, cenfunc="median", maxiters=5
            )
            clipped_valid = clipped.data[~clipped.mask]

            clipped_mean_flux = np.mean(clipped_valid) / (pixelscale**2)
            clipped_median_flux = np.median(clipped_valid) / (pixelscale**2)

            clipped_std_flux = (
                np.std(clipped_valid) if len(clipped_valid) > 1 else np.nan
            )

        flux_stats.append(
            {
                "Centre_pixel": (x, y),
                "Inner_radius_pix": r_inner,
                "Outer_radius_pix": r_outer,
                "Inner_radius_arcsec": r_inner * pixelscale,
                "Outer_radius_arcsec": r_outer * pixelscale,
                "SMA_annulus_centre_pix": (r_inner + r_outer) / 2,
                "SMA_annulus_centre_arcsec": (
                    (r_inner * pixelscale) + (r_outer * pixelscale)
                )
                / 2,
                "Mean_flux_annulus": mean_flux,
                "Median_flux_annulus": median_flux,
                "Std_flux_annulus": std_flux,
                "Total_valid_pix_annulus": total_valid,
                "Total_masked_pix_annulus": total_masked,
                "Clipped_mean_flux_annulus": clipped_mean_flux,
                "Clipped_median_flux_annulus": clipped_median_flux,
                "Std_clipped_flux_annulus": clipped_std_flux,
            }
        )

    if plot_filename:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(
            image,
            origin="lower",
            cmap="gray",
            norm=simple_norm(masked_image, "asinh", percent=80),
        )
        ax.imshow(
            mask.astype(float), origin="lower", cmap="Reds", vmin=0, vmax=1, alpha=0.8
        )
        for i in range(len(selected_annuli) - 1):
            r_outer = selected_annuli[i + 1]
            e = ellip[i]
            theta = np.deg2rad(pa[i])
            if annuli_shape == "circular":
                circle = Circle(
                    (x, y), radius=r_outer, edgecolor="blue", facecolor="none", lw=1
                )
                ax.add_patch(circle)
            else:
                ellipse = Ellipse(
                    (x, y),
                    width=2 * r_outer,
                    height=2 * r_outer * (1 - e),
                    angle=np.rad2deg(theta - np.pi / 2),
                    edgecolor="red",
                    facecolor="none",
                    lw=0.5,
                )
                ax.add_patch(ellipse)
        plt.savefig(plot_filename)
        plt.close()

    flux_stats_df = pd.DataFrame(flux_stats)
    return flux_stats_df, profile, problematic_annuli